# Train dan Inference DreamBooth LoRA + KronA

Notebook ini menjalankan eksperimen lengkap untuk dua adapter berbeda:

1. Train LoRA untuk semua dataset human dan logo, lalu inference dan grid visualisasi.
2. Train KronA/LoKr untuk semua dataset human dan logo, lalu inference dan grid visualisasi.

Output disimpan terstruktur berdasarkan metode, kategori dataset, nama dataset, learning rate, checkpoint, dan prompt. Gambar inference tidak ditampilkan di notebook agar proses batch tetap rapi.


## 1. Import Library


In [ ]:
import csv
import gc
import math
import os
import re
import subprocess
import time
from datetime import datetime
from pathlib import Path

import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw, ImageOps
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm
from transformers import CLIPTextModel, CLIPTokenizer

from diffusers import AutoencoderKL, DDPMScheduler, DiffusionPipeline, DPMSolverMultistepScheduler, UNet2DConditionModel
from peft import LoKrConfig, PeftModel, get_peft_model


## 2. Setup Eksperimen


In [ ]:
# Root project harus berada di folder repository diffusers.
PROJECT_ROOT = Path.cwd()
DATASET_DIR = PROJECT_ROOT / "dataset"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
SUMMARY_DIR = OUTPUT_ROOT / "summary"
SUMMARY_CSV = SUMMARY_DIR / "summary.csv"

MODEL_NAME = "runwayml/stable-diffusion-v1-5"
LEARNING_RATES = [1e-4, 5e-5]

MAX_TRAIN_STEPS = 2000
CHECKPOINTING_STEPS = MAX_TRAIN_STEPS // 5
CHECKPOINT_STEPS = list(range(CHECKPOINTING_STEPS, MAX_TRAIN_STEPS + 1, CHECKPOINTING_STEPS))
VALIDATION_EPOCHS = 15

MAX_TRAIN_STEPS_LOGO = 250
CHECKPOINTING_STEPS_LOGO = MAX_TRAIN_STEPS_LOGO // 5
CHECKPOINT_STEPS_LOGO = list(range(CHECKPOINTING_STEPS_LOGO, MAX_TRAIN_STEPS_LOGO + 1, CHECKPOINTING_STEPS_LOGO))
VALIDATION_EPOCHS_LOGO = 60

RESOLUTION = 512
TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
SEED = 42
RANK = 16

IMAGE_SIZE = 512
NUM_INFERENCE_STEPS = 40
GUIDANCE_SCALE = 7.5

# Ubah ke False jika ingin memaksa training ulang walaupun output sudah lengkap.
SKIP_EXISTING_TRAINING = True

# Ubah ke True untuk menjalankan smoke test cepat di satu dataset dan satu learning rate.
SMOKE_TEST_MODE = False
SMOKE_MAX_TRAIN_STEPS = 10
SMOKE_CHECKPOINTING_STEPS = 2


## 3. Dataset, Prompt, dan Struktur Folder


In [ ]:
def slugify(value):
    value = value.strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_") or "dataset"


def lr_slug(learning_rate):
    return f"lr_{learning_rate:.0e}".replace("e-0", "e-").replace("e+0", "e+")


def list_image_files(folder):
    image_exts = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
    return sorted([p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in image_exts])


def dataset_category(dataset_name):
    if dataset_name.startswith("human_"):
        return "human"
    if dataset_name.startswith("logo_"):
        return "logo"
    return None


def human_identity(dataset_name):
    if dataset_name.startswith("human_abiyamf"):
        return "human_abiyamf"
    if dataset_name.startswith("human_harsyit"):
        return "human_harsyit"
    return dataset_name

def extract_pure_name(dataset_name):
    parts = dataset_name.split('_')
    if len(parts) > 1:
        return parts[1]
    return dataset_name

def experiment_identity(dataset_name, category):
    pure_name = extract_pure_name(dataset_name)

    if category == "human":
        identity = pure_name
        identity_token = f"sks {identity} man"
        instance_prompt = f"a photo of {identity_token}"
        
    elif category == "logo":
        identity = pure_name
        identity_token = f"sks {identity}"
        instance_prompt = f"a {identity_token} icon logo"
        
    else:
        raise ValueError(f"Kategori dataset tidak dikenali: {dataset_name}")
    return identity, identity_token, instance_prompt


def build_prompts(identity_token, category):
    if category == "human":
        return [
            f"{identity_token} sitting in a coffee shop, photorealistic, natural window light, 85mm lens, f/1.8, natural skin texture, sharp focus, high detail, bokeh background",
            f"a beautiful watercolor portrait of {identity_token}, sharp clear face features, stylized watercolor painting style, majestic mountains background, soft color washes, elegant composition, visible paper texture",
            f"closeup professional corporate headshot of {identity_token} wearing a tailored formal suit, studio lighting, soft key light, neutral gray background, 85mm portrait lens, sharp focus on eyes, highly detailed skin texture, photorealistic",
        ]
        
    if category == "logo":
        return [
            f"minimalist design of {identity_token} icon logo, soft pastel colour palette, clean studio lighting, sharp vector-like edges, realistic texture",
            f"8-bit video game graphics of {identity_token} icon logo, clean studio lighting, sharp vector-like edges, realistic texture",
            f"modern watercolor of {identity_token} icon logo, centered composition, crisp edges, clean paper texture background, minimal design, high detail",
        ]
        
    raise ValueError(f"Kategori prompt tidak dikenali: {category}")


NEGATIVE_PROMPT = (
    "low quality, blurry, distorted, deformed, bad anatomy, extra fingers, "
    "cartoon, painting, watermark, text artifacts, cropped logo, misspelled text"
)


def discover_datasets(dataset_dir=DATASET_DIR):
    if not dataset_dir.is_dir():
        raise FileNotFoundError(f"Folder dataset tidak ditemukan: {dataset_dir}")

    datasets = []
    for folder in sorted(dataset_dir.iterdir()):
        if not folder.is_dir():
            continue
        category = dataset_category(folder.name)
        if category is None:
            continue
        image_paths = list_image_files(folder)
        if not image_paths:
            print(f"Lewati {folder.name}: tidak ada file gambar.")
            continue
        identity, identity_token, instance_prompt = experiment_identity(folder.name, category)
        datasets.append(
            {
                "name": folder.name,
                "slug": slugify(folder.name),
                "category": category,
                "path": folder,
                "image_paths": image_paths,
                "identity": identity,
                "identity_token": identity_token,
                "instance_prompt": instance_prompt,
                "prompts": build_prompts(identity_token, category),
            }
        )
    return datasets


DATASETS = discover_datasets()
if SMOKE_TEST_MODE:
    DATASETS = DATASETS[:1]
    LEARNING_RATES = LEARNING_RATES[:1]
    MAX_TRAIN_STEPS = SMOKE_MAX_TRAIN_STEPS
    CHECKPOINTING_STEPS = SMOKE_CHECKPOINTING_STEPS
    CHECKPOINT_STEPS = list(range(CHECKPOINTING_STEPS, MAX_TRAIN_STEPS + 1, CHECKPOINTING_STEPS))
    VALIDATION_EPOCHS = 1


def max_train_steps_for_dataset(dataset):
    if SMOKE_TEST_MODE:
        return MAX_TRAIN_STEPS
    if dataset["category"] == "logo":
        return MAX_TRAIN_STEPS_LOGO
    return MAX_TRAIN_STEPS


def checkpointing_steps_for_dataset(dataset):
    if SMOKE_TEST_MODE:
        return CHECKPOINTING_STEPS
    if dataset["category"] == "logo":
        return CHECKPOINTING_STEPS_LOGO
    return CHECKPOINTING_STEPS


def checkpoint_steps_for_dataset(dataset):
    max_train_steps = max_train_steps_for_dataset(dataset)
    checkpointing_steps = checkpointing_steps_for_dataset(dataset)
    return list(range(checkpointing_steps, max_train_steps + 1, checkpointing_steps))


def validation_epochs_for_dataset(dataset):
    if SMOKE_TEST_MODE:
        return VALIDATION_EPOCHS
    if dataset["category"] == "logo":
        return VALIDATION_EPOCHS_LOGO
    return VALIDATION_EPOCHS


print("Dataset yang akan diproses:")
for dataset in DATASETS:
    print(f"- {dataset['category']:5s} | {dataset['name']} | {len(dataset['image_paths'])} gambar | {max_train_steps_for_dataset(dataset)} steps | checkpoints: {checkpoint_steps_for_dataset(dataset)} | prompt: {dataset['instance_prompt']}")

print(f"\nLearning rates: {LEARNING_RATES}")


## 4. Helper Output, Inference, dan Grid


In [ ]:
def ensure_dir(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def method_root(method):
    return ensure_dir(OUTPUT_ROOT / method)


def summary_dir():
    return ensure_dir(SUMMARY_DIR)


def append_summary_row(stage, method, dataset, learning_rate, start_time, end_time, status, output_path, checkpoint_step="", note=""):
    """Append log durasi training/inference ke outputs/summary/summary.csv."""
    summary_dir()
    fieldnames = [
        "timestamp",
        "date",
        "time",
        "stage",
        "method",
        "category",
        "dataset",
        "learning_rate",
        "learning_rate_slug",
        "checkpoint_step",
        "max_train_steps",
        "checkpoint_steps",
        "start_time",
        "end_time",
        "duration_seconds",
        "duration_minutes",
        "status",
        "output_path",
        "note",
    ]
    timestamp = end_time.strftime("%Y-%m-%d %H:%M:%S")
    duration_seconds = (end_time - start_time).total_seconds()
    checkpoint_steps = checkpoint_steps_for_dataset(dataset)
    row = {
        "timestamp": timestamp,
        "date": end_time.strftime("%Y-%m-%d"),
        "time": end_time.strftime("%H:%M:%S"),
        "stage": stage,
        "method": method,
        "category": dataset["category"],
        "dataset": dataset["name"],
        "learning_rate": learning_rate,
        "learning_rate_slug": lr_slug(learning_rate),
        "checkpoint_step": checkpoint_step,
        "max_train_steps": max_train_steps_for_dataset(dataset),
        "checkpoint_steps": "|".join(str(step) for step in checkpoint_steps),
        "start_time": start_time.strftime("%Y-%m-%d %H:%M:%S"),
        "end_time": timestamp,
        "duration_seconds": round(duration_seconds, 3),
        "duration_minutes": round(duration_seconds / 60, 3),
        "status": status,
        "output_path": str(output_path),
        "note": note,
    }
    file_exists = SUMMARY_CSV.is_file()
    with SUMMARY_CSV.open("a", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def train_dir(method, dataset, learning_rate):
    return method_root(method) / "train" / dataset["category"] / dataset["slug"] / lr_slug(learning_rate)


def inference_dir(method, dataset, learning_rate, checkpoint_step):
    return method_root(method) / "inference" / dataset["category"] / dataset["slug"] / lr_slug(learning_rate) / f"checkpoint-{checkpoint_step}"


def grid_dir(method, dataset, learning_rate):
    return method_root(method) / "grids" / dataset["category"] / dataset["slug"] / lr_slug(learning_rate)


def checkpoint_dir(method, dataset, learning_rate, checkpoint_step):
    return train_dir(method, dataset, learning_rate) / f"checkpoint-{checkpoint_step}"


def training_output_complete(method, dataset, learning_rate):
    return all(checkpoint_dir(method, dataset, learning_rate, step).is_dir() for step in checkpoint_steps_for_dataset(dataset))


def cleanup_torch():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_base_pipeline():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32

    pipe = DiffusionPipeline.from_pretrained(
        MODEL_NAME,
        torch_dtype=dtype,
        use_safetensors=True,
    )
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    pipe.to(device)

    if device == "cuda":
        pipe.enable_attention_slicing()
        pipe.enable_vae_slicing()
        try:
            pipe.enable_xformers_memory_efficient_attention()
        except Exception as exc:
            print(f"xformers tidak aktif: {exc}")
    return pipe, device


def generate_prompt_images(pipe, device, prompts, output_folder):
    ensure_dir(output_folder)
    output_paths = []
    for prompt_index, prompt in enumerate(prompts, start=1):
        generator = torch.Generator(device=device).manual_seed(SEED + prompt_index)
        image = pipe(
            prompt=prompt,
            negative_prompt=NEGATIVE_PROMPT,
            width=IMAGE_SIZE,
            height=IMAGE_SIZE,
            num_inference_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            generator=generator,
        ).images[0]
        output_path = Path(output_folder) / f"prompt_{prompt_index}.png"
        image.save(output_path)
        output_paths.append(output_path)
    return output_paths


def create_contact_sheet(image_paths, size=IMAGE_SIZE):
    image_paths = list(image_paths)
    if not image_paths:
        raise ValueError("Tidak ada gambar referensi untuk contact sheet.")

    count = len(image_paths)
    cols = math.ceil(math.sqrt(count))
    rows = math.ceil(count / cols)
    cell_w = size // cols
    cell_h = size // rows
    sheet = Image.new("RGB", (size, size), "white")

    for index, image_path in enumerate(image_paths):
        with Image.open(image_path) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
            image = ImageOps.fit(image, (cell_w, cell_h), method=Image.Resampling.LANCZOS)
        x = (index % cols) * cell_w
        y = (index // cols) * cell_h
        sheet.paste(image, (x, y))
    return sheet


def placeholder_image(label, size=IMAGE_SIZE):
    image = Image.new("RGB", (size, size), (245, 245, 245))
    draw = ImageDraw.Draw(image)
    lines = label.split("\n")
    line_height = 18
    y = (size - line_height * len(lines)) // 2
    for line in lines:
        bbox = draw.textbbox((0, 0), line)
        x = (size - (bbox[2] - bbox[0])) // 2
        draw.text((x, y), line, fill=(90, 90, 90))
        y += line_height
    return image


def add_caption(image, caption, caption_height=56):
    image = image.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE + caption_height), "white")
    canvas.paste(image, (0, caption_height))
    draw = ImageDraw.Draw(canvas)
    bbox = draw.textbbox((0, 0), caption)
    text_w = bbox[2] - bbox[0]
    draw.text(((IMAGE_SIZE - text_w) // 2, 18), caption, fill=(20, 20, 20))
    return canvas


def wrap_text(text, max_chars=48):
    words = text.split()
    lines = []
    current = ""
    for word in words:
        candidate = f"{current} {word}".strip()
        if len(candidate) <= max_chars:
            current = candidate
        else:
            if current:
                lines.append(current)
            current = word
    if current:
        lines.append(current)
    return lines


def prompt_panel(prompt, prompt_index, width=420, height=IMAGE_SIZE + 56):
    panel = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(panel)
    draw.text((18, 18), f"Prompt {prompt_index}", fill=(20, 20, 20))
    y = 56
    for line in wrap_text(prompt, max_chars=42):
        if y > height - 28:
            draw.text((18, y), "...", fill=(45, 45, 45))
            break
        draw.text((18, y), line, fill=(45, 45, 45))
        y += 24
    return panel


def make_comparison_grid(method, dataset, learning_rate):
    reference = create_contact_sheet(dataset["image_paths"])
    prompts = dataset["prompts"]
    checkpoint_steps = checkpoint_steps_for_dataset(dataset)
    caption_height = 56
    cell_h = IMAGE_SIZE + caption_height
    prompt_column_width = 420
    image_columns = 1 + len(checkpoint_steps)
    grid_width = prompt_column_width + IMAGE_SIZE * image_columns
    grid = Image.new("RGB", (grid_width, cell_h * len(prompts)), "white")

    for row_index, prompt in enumerate(prompts, start=1):
        row_y = (row_index - 1) * cell_h
        grid.paste(prompt_panel(prompt, row_index, width=prompt_column_width, height=cell_h), (0, row_y))

        row_images = [add_caption(reference, "training data")]
        for step in checkpoint_steps:
            image_path = inference_dir(method, dataset, learning_rate, step) / f"prompt_{row_index}.png"
            if image_path.is_file():
                with Image.open(image_path) as image:
                    generated = ImageOps.exif_transpose(image).convert("RGB")
            else:
                generated = placeholder_image(f"missing\ncheckpoint-{step}\nprompt_{row_index}")
            row_images.append(add_caption(generated, f"checkpoint-{step}"))

        for col_index, image in enumerate(row_images):
            x = prompt_column_width + col_index * IMAGE_SIZE
            grid.paste(image, (x, row_y))

    output_path = ensure_dir(grid_dir(method, dataset, learning_rate)) / "comparison_grid.png"
    grid.save(output_path)
    return output_path


summary_dir()
for method in ["LoRA", "Krona"]:
    for subdir in ["train", "inference", "grids"]:
        ensure_dir(method_root(method) / subdir)


# DreamBooth LoRA

Bagian ini memakai script resmi Diffusers `examples/dreambooth/train_dreambooth_lora.py`, mengikuti pola notebook LoRA referensi.


## 5. Training LoRA


In [ ]:
def train_lora_single_run(dataset, learning_rate):
    script_path = PROJECT_ROOT / "examples" / "dreambooth" / "train_dreambooth_lora.py"
    if not script_path.is_file():
        raise FileNotFoundError(f"Script training LoRA tidak ditemukan: {script_path}")

    output_dir = ensure_dir(train_dir("LoRA", dataset, learning_rate))
    start_time = datetime.now()
    if SKIP_EXISTING_TRAINING and training_output_complete("LoRA", dataset, learning_rate):
        print(f"[LoRA] Skip, checkpoint lengkap: {output_dir}")
        append_summary_row("training", "LoRA", dataset, learning_rate, start_time, datetime.now(), "skipped", output_dir, note="checkpoint lengkap")
        return output_dir

    max_train_steps = max_train_steps_for_dataset(dataset)
    checkpointing_steps = checkpointing_steps_for_dataset(dataset)
    validation_epochs = validation_epochs_for_dataset(dataset)

    command = [
        "accelerate",
        "launch",
        str(script_path),
        "--pretrained_model_name_or_path", MODEL_NAME,
        "--instance_data_dir", str(dataset["path"]),
        "--instance_prompt", dataset["instance_prompt"],
        "--resolution", str(RESOLUTION),
        "--train_batch_size", str(TRAIN_BATCH_SIZE),
        "--gradient_accumulation_steps", str(GRADIENT_ACCUMULATION_STEPS),
        "--learning_rate", str(learning_rate),
        "--lr_scheduler", "constant",
        "--lr_warmup_steps", "0",
        "--mixed_precision", "fp16",
        "--gradient_checkpointing",
        "--use_8bit_adam",
        "--rank", str(RANK),
        "--max_train_steps", str(max_train_steps),
        "--checkpointing_steps", str(checkpointing_steps),
        "--validation_prompt", f"{dataset['instance_prompt']}, soft studio lighting, high detail",
        "--validation_epochs", str(validation_epochs),
        "--seed", str(SEED),
        "--output_dir", str(output_dir),
    ]

    print(f"\n[LoRA] Training {dataset['name']} | {lr_slug(learning_rate)} | {max_train_steps} steps")
    print(" ".join(command))
    try:
        subprocess.run(command, check=True)
    except Exception as exc:
        append_summary_row("training", "LoRA", dataset, learning_rate, start_time, datetime.now(), "failed", output_dir, note=str(exc))
        raise

    append_summary_row("training", "LoRA", dataset, learning_rate, start_time, datetime.now(), "completed", output_dir)
    return output_dir


def train_all_lora():
    for dataset in DATASETS:
        for learning_rate in LEARNING_RATES:
            train_lora_single_run(dataset, learning_rate)
            cleanup_torch()


train_all_lora()


## 6. Inference LoRA dan Grid Visualisasi


In [ ]:
def run_lora_inference_single_run(dataset, learning_rate):
    print(f"\n[LoRA] Inference {dataset['name']} | {lr_slug(learning_rate)}")
    start_time = datetime.now()
    status = "completed"
    notes = []
    inference_root = method_root("LoRA") / "inference" / dataset["category"] / dataset["slug"] / lr_slug(learning_rate)
    grid_path = None

    try:
        for step in checkpoint_steps_for_dataset(dataset):
            weights_dir = checkpoint_dir("LoRA", dataset, learning_rate, step)
            if not weights_dir.is_dir():
                message = f"checkpoint-{step} belum ada"
                print(f"[LoRA] Lewati checkpoint yang belum ada: {weights_dir}")
                notes.append(message)
                status = "completed_with_missing_checkpoints"
                continue

            pipe, device = load_base_pipeline()
            pipe.load_lora_weights(str(weights_dir))
            output_folder = inference_dir("LoRA", dataset, learning_rate, step)
            generate_prompt_images(pipe, device, dataset["prompts"], output_folder)
            del pipe
            cleanup_torch()

        grid_path = make_comparison_grid("LoRA", dataset, learning_rate)
        print(f"[LoRA] Grid tersimpan: {grid_path}")
    except Exception as exc:
        append_summary_row("inference", "LoRA", dataset, learning_rate, start_time, datetime.now(), "failed", inference_root, note=str(exc))
        raise

    append_summary_row("inference", "LoRA", dataset, learning_rate, start_time, datetime.now(), status, inference_root, note="; ".join(notes))
    return grid_path


def run_all_lora_inference():
    for dataset in DATASETS:
        for learning_rate in LEARNING_RATES:
            run_lora_inference_single_run(dataset, learning_rate)


run_all_lora_inference()


# DreamBooth KronA / LoKr

Bagian ini mengikuti pola `train_new_KronA.ipynb`: model dasar dimuat manual, UNet diberi adapter `LoKrConfig`, lalu hanya parameter adapter yang dilatih dan disimpan.


## 7. Dataset dan Training KronA


In [ ]:
class DreamBoothDataset(Dataset):
    def __init__(self, instance_data_root, instance_prompt, tokenizer, size=512):
        self.instance_data_root = Path(instance_data_root)
        self.instance_images_path = list_image_files(self.instance_data_root)
        if not self.instance_images_path:
            raise ValueError(f"Tidak ada gambar training di {self.instance_data_root}")
        self.instance_prompt = instance_prompt
        self.tokenizer = tokenizer
        self.size = size
        self.image_transforms = transforms.Compose(
            [
                transforms.Resize(size, interpolation=transforms.InterpolationMode.BILINEAR),
                transforms.CenterCrop(size),
                transforms.ToTensor(),
                transforms.Normalize([0.5], [0.5]),
            ]
        )

    def __len__(self):
        return len(self.instance_images_path)

    def __getitem__(self, index):
        image_path = self.instance_images_path[index % len(self.instance_images_path)]
        image = Image.open(image_path).convert("RGB")
        example = {
            "pixel_values": self.image_transforms(image),
            "input_ids": self.tokenizer(
                self.instance_prompt,
                truncation=True,
                padding="max_length",
                max_length=self.tokenizer.model_max_length,
                return_tensors="pt",
            ).input_ids[0],
        }
        return example


def save_krona_checkpoint(unet, output_dir, global_step):
    checkpoint_output_dir = ensure_dir(Path(output_dir) / f"checkpoint-{global_step}")
    unet.save_pretrained(checkpoint_output_dir)
    print(f"[Krona] Checkpoint tersimpan: {checkpoint_output_dir}")


def train_krona_single_run(dataset, learning_rate):
    output_dir = ensure_dir(train_dir("Krona", dataset, learning_rate))
    start_time = datetime.now()
    if SKIP_EXISTING_TRAINING and training_output_complete("Krona", dataset, learning_rate):
        print(f"[Krona] Skip, checkpoint lengkap: {output_dir}")
        append_summary_row("training", "Krona", dataset, learning_rate, start_time, datetime.now(), "skipped", output_dir, note="checkpoint lengkap")
        return output_dir

    max_train_steps = max_train_steps_for_dataset(dataset)
    checkpoint_steps = checkpoint_steps_for_dataset(dataset)

    try:
        print(f"\n[Krona] Training {dataset['name']} | {lr_slug(learning_rate)} | {max_train_steps} steps")
        torch.manual_seed(SEED)
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        weight_dtype = torch.float16 if device.type == "cuda" else torch.float32

        tokenizer = CLIPTokenizer.from_pretrained(MODEL_NAME, subfolder="tokenizer")
        text_encoder = CLIPTextModel.from_pretrained(MODEL_NAME, subfolder="text_encoder", torch_dtype=weight_dtype)
        vae = AutoencoderKL.from_pretrained(MODEL_NAME, subfolder="vae", torch_dtype=weight_dtype)
        unet = UNet2DConditionModel.from_pretrained(MODEL_NAME, subfolder="unet", torch_dtype=weight_dtype)
        noise_scheduler = DDPMScheduler.from_pretrained(MODEL_NAME, subfolder="scheduler")

        vae.requires_grad_(False)
        text_encoder.requires_grad_(False)
        unet.requires_grad_(False)

        vae.to(device)
        text_encoder.to(device)
        unet.to(device)
        vae.eval()
        text_encoder.eval()

        if hasattr(unet, "enable_gradient_checkpointing"):
            unet.enable_gradient_checkpointing()

        lokr_config = LoKrConfig(
            r=RANK,
            alpha=RANK * 2,
            target_modules=["to_q", "to_k", "to_v", "to_out.0", "proj_in", "proj_out"],
            use_effective_conv2d=True,
            decompose_both=True,
            decompose_factor=-1,
        )
        unet = get_peft_model(unet, lokr_config)
        unet.to(device)
        unet.print_trainable_parameters()

        train_dataset = DreamBoothDataset(
            instance_data_root=dataset["path"],
            instance_prompt=dataset["instance_prompt"],
            tokenizer=tokenizer,
            size=RESOLUTION,
        )
        train_dataloader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True)
        trainable_params = [param for param in unet.parameters() if param.requires_grad]
        optimizer = torch.optim.AdamW(trainable_params, lr=learning_rate, weight_decay=1e-2)

        unet.train()
        optimizer.zero_grad(set_to_none=True)
        global_step = 0
        accumulation_counter = 0
        progress_bar = tqdm(total=max_train_steps, desc=f"Krona {dataset['slug']} {lr_slug(learning_rate)}")

        while global_step < max_train_steps:
            for batch in train_dataloader:
                with torch.no_grad():
                    latents = vae.encode(batch["pixel_values"].to(device, dtype=weight_dtype)).latent_dist.sample()
                    latents = latents * vae.config.scaling_factor

                noise = torch.randn_like(latents)
                batch_size = latents.shape[0]
                timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (batch_size,), device=device).long()
                noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
                with torch.no_grad():
                    encoder_hidden_states = text_encoder(batch["input_ids"].to(device))[0]
                model_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample

                if noise_scheduler.config.prediction_type == "epsilon":
                    target = noise
                elif noise_scheduler.config.prediction_type == "v_prediction":
                    target = noise_scheduler.get_velocity(latents, noise, timesteps)
                else:
                    raise ValueError(f"prediction_type tidak didukung: {noise_scheduler.config.prediction_type}")

                loss = F.mse_loss(model_pred.float(), target.float(), reduction="mean")
                (loss / GRADIENT_ACCUMULATION_STEPS).backward()
                accumulation_counter += 1

                if accumulation_counter % GRADIENT_ACCUMULATION_STEPS == 0:
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
                    global_step += 1
                    progress_bar.update(1)
                    progress_bar.set_postfix({"loss": loss.detach().item(), "lr": learning_rate})

                    if global_step in checkpoint_steps:
                        save_krona_checkpoint(unet, output_dir, global_step)

                    if global_step >= max_train_steps:
                        break

        progress_bar.close()

        if max_train_steps not in checkpoint_steps:
            save_krona_checkpoint(unet, output_dir, max_train_steps)

        final_dir = ensure_dir(output_dir / "final")
        unet.save_pretrained(final_dir)
        print(f"[Krona] Final adapter tersimpan: {final_dir}")

        del tokenizer, text_encoder, vae, unet, noise_scheduler, optimizer, train_dataset, train_dataloader
        cleanup_torch()
    except Exception as exc:
        append_summary_row("training", "Krona", dataset, learning_rate, start_time, datetime.now(), "failed", output_dir, note=str(exc))
        raise

    append_summary_row("training", "Krona", dataset, learning_rate, start_time, datetime.now(), "completed", output_dir)
    return output_dir


def train_all_krona():
    for dataset in DATASETS:
        for learning_rate in LEARNING_RATES:
            train_krona_single_run(dataset, learning_rate)
            cleanup_torch()


train_all_krona()


## 8. Inference KronA dan Grid Visualisasi


In [ ]:
def run_krona_inference_single_run(dataset, learning_rate):
    print(f"\n[Krona] Inference {dataset['name']} | {lr_slug(learning_rate)}")
    start_time = datetime.now()
    status = "completed"
    notes = []
    inference_root = method_root("Krona") / "inference" / dataset["category"] / dataset["slug"] / lr_slug(learning_rate)
    grid_path = None

    try:
        for step in checkpoint_steps_for_dataset(dataset):
            weights_dir = checkpoint_dir("Krona", dataset, learning_rate, step)
            if not weights_dir.is_dir():
                message = f"checkpoint-{step} belum ada"
                print(f"[Krona] Lewati checkpoint yang belum ada: {weights_dir}")
                notes.append(message)
                status = "completed_with_missing_checkpoints"
                continue

            pipe, device = load_base_pipeline()
            pipe.unet = PeftModel.from_pretrained(pipe.unet, str(weights_dir))
            pipe.to(device)
            output_folder = inference_dir("Krona", dataset, learning_rate, step)
            generate_prompt_images(pipe, device, dataset["prompts"], output_folder)
            del pipe
            cleanup_torch()

        grid_path = make_comparison_grid("Krona", dataset, learning_rate)
        print(f"[Krona] Grid tersimpan: {grid_path}")
    except Exception as exc:
        append_summary_row("inference", "Krona", dataset, learning_rate, start_time, datetime.now(), "failed", inference_root, note=str(exc))
        raise

    append_summary_row("inference", "Krona", dataset, learning_rate, start_time, datetime.now(), status, inference_root, note="; ".join(notes))
    return grid_path


def run_all_krona_inference():
    for dataset in DATASETS:
        for learning_rate in LEARNING_RATES:
            run_krona_inference_single_run(dataset, learning_rate)


run_all_krona_inference()


## 9. Ringkasan Output


In [ ]:
def summarize_expected_outputs():
    print("Ringkasan lokasi output:")
    print(f"Summary CSV append-only: {SUMMARY_CSV}")
    for method in ["LoRA", "Krona"]:
        print(f"\n{method}")
        for dataset in DATASETS:
            for learning_rate in LEARNING_RATES:
                print(f"- {dataset['category']}/{dataset['slug']}/{lr_slug(learning_rate)}")
                print(f"  train    : {train_dir(method, dataset, learning_rate)}")
                print(f"  inference: {method_root(method) / 'inference' / dataset['category'] / dataset['slug'] / lr_slug(learning_rate)}")
                print(f"  grid     : {grid_dir(method, dataset, learning_rate) / 'comparison_grid.png'}")


summarize_expected_outputs()
